In [ ]:
import torch
import numpy as np
import copy

from sklearn.datasets import load_diabetes
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, root_mean_squared_error, mean_squared_error, mean_absolute_error
from torch import Tensor
from torch.nn import Sequential, Linear, ReLU, MSELoss, Dropout
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [33]:
# Processar dados em CUDA (GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"O processamento será feito em {device}")

O processamento será feito em cuda


In [ ]:
# Carregamento Dados
dataset = load_diabetes()

X = dataset.data
y = np.expand_dims(dataset.target, axis = 1)

# Configurações Gerais
batch_size = 8
max_epochs = 500
k_folds = 5 # divide os dados em 5 partes
kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

# Listas para guardar o resultado de cada Fold (dobra/iteração)
resultados_r2 = []
resultados_rmse = []
resultados_mse = []
resultados_mae = []

# Função para criar modelo "zerado" a cada Fold
def criar_modelo():
    # Rede + Regularização (Dropout)
    model = Sequential(
        Linear(10, 32),
        ReLU(),
        Dropout(0.2),
        Linear(32,16),
        ReLU(),
        Linear(16,1),
    ).to(device)

    optimizer = Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
    criterion = MSELoss()
    scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=15)
    
    return model, optimizer, criterion, scheduler

print(f"Iniciando K-Fold Cross Validation com {k_folds} folds...\n")

Iniciando K-Fold Cross Validation com 5 folds...



In [ ]:
# LOOP DO K-FOLD
for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    print(f"--- Treinando Fold {fold + 1}/{k_folds} ---")

    # Separar os dados da rodada (fold) atual
    X_treino, X_teste = X[train_idx], X[test_idx]
    y_treino, y_teste = y[train_idx], y[test_idx]

    # Normalização independente para cada fold (assim esse fold não sabe a média de todos os dados, somente desta parcela / evita memory leak)
    scaler_y = StandardScaler()
    y_treino_scaled = scaler_y.fit_transform(y_treino)
    y_teste_scaled = scaler_y.transform(y_teste)

    # DataLoaders da rodada
    train_loader = DataLoader(TensorDataset(Tensor(X_treino), Tensor(y_treino_scaled)), batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(TensorDataset(Tensor(X_teste), Tensor(y_teste_scaled)), batch_size=batch_size, shuffle=False)

    # Recriando modelo, otimizador, criterion e scheduler do zero para este fold
    model, optimizer, criterion, scheduler = criar_modelo()

    best_loss = float('inf')
    patience_counter = 0
    patience_limit = 40
    best_model_weights = copy.deepcopy(model.state_dict())


    # LOOP DE TREINAMENTO EARLY STOPPING
    for epoch in range(max_epochs):
        model.train()
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            output = model(batch_x)
            loss = criterion(output, batch_y)
            loss.backward()
            optimizer.step()

        # Validação
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_x, batch_y in test_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                output = model(batch_x)
                val_loss = val_loss + criterion(output, batch_y).item() * batch_x.size(0)

        if val_loss < best_loss:
            best_loss = val_loss
            best_model_weights = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter = patience_counter + 1

        if patience_counter >= patience_limit:
            print(f"   Early Stopping na época {epoch + 1}\n")
            break

    # Restaura os melhores pesos da rodada
    model.load_state_dict(best_model_weights)


    # 4. Avaliação Final do Fold
    model.eval()
    with torch.no_grad():
        X_teste_tensor = Tensor(X_teste).to(device)
        output_scaled = model(X_teste_tensor).cpu().numpy()
        output_real = scaler_y.inverse_transform(output_scaled)

        r2 = r2_score(y_teste, output_real)
        mse = mean_squared_error(y_teste, output_real)
        rmse = root_mean_squared_error(y_teste, output_real)
        mae = mean_absolute_error(y_teste, output_real)

        resultados_r2.append(r2)
        resultados_mse.append(mse)
        resultados_rmse.append(rmse)
        resultados_mae.append(mae)


--- Treinando Fold 1/5 ---
   Early Stopping na época 146

--- Treinando Fold 2/5 ---
   Early Stopping na época 48

--- Treinando Fold 3/5 ---
   Early Stopping na época 90

--- Treinando Fold 4/5 ---
   Early Stopping na época 60

--- Treinando Fold 5/5 ---
   Early Stopping na época 47



In [36]:
# MÉDIA FINAL
print("-" * 30)
print("MÉDIA FINAL DA VALIDAÇÃO CRUZADA:")
print(f"R^2 Médio: {np.mean(resultados_r2):.4} (+/- {np.std(resultados_r2):.4f})")
print(f"RMSE Médio: {np.mean(resultados_rmse):.4} (+/- {np.std(resultados_rmse):.4f})")

------------------------------
MÉDIA FINAL DA VALIDAÇÃO CRUZADA:
R^2 Médio: 0.5096 (+/- 0.0762)
RMSE Médio: 53.26 (+/- 2.9069)
